imports + PASTA_RAW

In [14]:
import sys
import json
from pathlib import Path
from datetime import datetime, timezone, timedelta

import pandas as pd
import requests

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    PASTA_PROJETO = Path("/content/drive/MyDrive/ripe_atlas")
else:
    PASTA_PROJETO = Path("./ripe_atlas")

PASTA_RAW = PASTA_PROJETO / "raw"
PASTA_RAW.mkdir(parents=True, exist_ok=True)

print("Arquivos raw serão salvos em:", PASTA_RAW)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Arquivos raw serão salvos em: /content/drive/MyDrive/ripe_atlas/raw


0. Pré-requisito

In [15]:
# Variáveis herdadas da etapa de descoberta
measurement_id = 28095697
anchor_fqdn = "br-bhz-as10417-client.anchors.atlas.ripe.net"
anchor_probe_ids = [
    6251, 6313, 6316, 6317, 6320, 6323, 6324, 6325, 6326, 6327,
    6328, 6329, 6330, 6331, 6332, 6333, 6334, 6335, 6336, 6342
]

def obter_metadados_medicao(measurement_id, timeout_requisicao):
    url_metadados = f"https://atlas.ripe.net/api/v2/measurements/{measurement_id}/"
    response = requests.get(url_metadados, timeout=timeout_requisicao)
    response.raise_for_status()
    info = response.json()
    return {
        "is_oneoff": info.get("is_oneoff"),
        "interval": info.get("interval"),
        "packets": info.get("packets"),
        "start_time": info.get("start_time"),
        "stop_time": info.get("stop_time"),
    }

1. Parâmetros e verificação de que a medição é contínua

In [16]:
# Defina o ID da sua medição do RIPE Atlas aqui
measurement_id = 28095697  # Substitua pelo ID real da sua medição

JANELA_HORAS = 168
TIMEOUT_REQUISICAO = 60
FORMATO_TABELAR = "csv"

metadados_medicao = obter_metadados_medicao(measurement_id, TIMEOUT_REQUISICAO)
print("is_oneoff:", metadados_medicao["is_oneoff"])
print("interval:", metadados_medicao["interval"])

assert metadados_medicao["is_oneoff"] is False, (
    "Essa medição é one-off — não serve para coletar pares ao longo do tempo."
)

is_oneoff: False
interval: 240


2. Coletar os resultados dos pares (uma chamada só)

In [17]:
def coletar_resultados_pares(measurement_id, prb_ids, janela_horas, timeout_requisicao, pasta_raw, formato_tabelar="csv"):
    fim_utc = datetime.now(timezone.utc)
    inicio_utc = fim_utc - timedelta(hours=janela_horas)

    url = f"https://atlas.ripe.net/api/v2/measurements/{measurement_id}/results/"
    parametros = {
        "start": int(inicio_utc.timestamp()),
        "stop": int(fim_utc.timestamp()),
        "probe_ids": ",".join(str(p) for p in prb_ids),
        "format": "json",
    }

    print("URL:", url)
    print("Início (UTC):", inicio_utc.isoformat())
    print("Fim (UTC):", fim_utc.isoformat())
    print("Pares (prb_id) solicitados:", len(prb_ids))

    instante_requisicao_utc = datetime.now(timezone.utc)
    response = requests.get(url, params=parametros, timeout=timeout_requisicao)
    response.raise_for_status()
    dados_json = response.json()

    print("Status HTTP:", response.status_code)
    print("Quantidade de registros recebidos:", len(dados_json))

    if len(dados_json) == 0:
        raise RuntimeError("Nenhum registro retornado — revise a janela ou os prb_ids.")

    df_raw = pd.json_normalize(dados_json, sep="_")
    display(df_raw.head())

    timestamp_arquivo = instante_requisicao_utc.strftime("%Y%m%dT%H%M%SZ")
    nome_base = f"ripe_atlas_pares_m{measurement_id}_{timestamp_arquivo}"

    caminho_json = pasta_raw / f"{nome_base}.json"
    with caminho_json.open("w", encoding="utf-8") as arquivo:
        json.dump(dados_json, arquivo, ensure_ascii=False, indent=2)

    caminho_tabela = pasta_raw / f"{nome_base}.{formato_tabelar}"
    if formato_tabelar == "csv":
        df_raw.to_csv(caminho_tabela, index=False, encoding="utf-8")
    else:
        df_raw.to_parquet(caminho_tabela, index=False)

    metadados = {
        "measurement_id": measurement_id,
        "is_oneoff": metadados_medicao["is_oneoff"],
        "interval": metadados_medicao["interval"],
        "quantidade_pares_solicitados": len(prb_ids),
        "pares_prb_ids": list(prb_ids),
        "url": url,
        "parametros": parametros,
        "inicio_utc": inicio_utc.isoformat(),
        "fim_utc": fim_utc.isoformat(),
        "coletado_em_utc": instante_requisicao_utc.isoformat(),
        "status_http": response.status_code,
        "quantidade_registros": len(dados_json),
        "arquivo_json": caminho_json.name,
        "arquivo_tabelar": caminho_tabela.name,
    }
    caminho_metadados = pasta_raw / f"{nome_base}_metadata.json"
    with caminho_metadados.open("w", encoding="utf-8") as arquivo:
        json.dump(metadados, arquivo, ensure_ascii=False, indent=2)

    return {"df_raw": df_raw, "dados_json": dados_json, "caminho_json": caminho_json,
            "caminho_tabela": caminho_tabela, "caminho_metadados": caminho_metadados}


resultado_pares = coletar_resultados_pares(
    measurement_id, anchor_probe_ids[:20], JANELA_HORAS, TIMEOUT_REQUISICAO, PASTA_RAW
)

URL: https://atlas.ripe.net/api/v2/measurements/28095697/results/
Início (UTC): 2026-10-01T13:53:39.407734+00:00
Fim (UTC): 2026-10-08T13:53:39.407734+00:00
Pares (prb_id) solicitados: 20
Status HTTP: 200
Quantidade de registros recebidos: 50251


,fw,mver,lts,dst_name,af,dst_addr,src_addr,proto,ttl,size,...,avg,msm_id,prb_id,timestamp,msm_name,from,type,group_id,step,stored_timestamp
0,5130,2.6.4,43,150.164.1.222,4,150.164.1.222,193.65.45.86,ICMP,42.0,64,...,226.429704,28095697,6251,1790862952,Ping,193.65.45.86,ping,28095696,240,1790863034
1,5130,2.6.4,49,150.164.1.222,4,150.164.1.222,185.138.143.34,ICMP,44.0,64,...,210.708915,28095697,6313,1790862956,Ping,185.138.143.34,ping,28095696,240,1790862966
2,5130,2.6.4,46,150.164.1.222,4,150.164.1.222,213.212.129.68,ICMP,44.0,64,...,224.548937,28095697,6316,1790862954,Ping,213.212.129.68,ping,28095696,240,1790862984
3,5130,2.6.4,34,150.164.1.222,4,150.164.1.222,130.59.80.2,ICMP,47.0,64,...,135.075325,28095697,6317,1790862955,Ping,130.59.80.2,ping,28095696,240,1790862979
4,5130,2.6.4,24,150.164.1.222,4,150.164.1.222,188.172.218.22,ICMP,49.0,64,...,224.192610,28095697,6329,1790862959,Ping,188.172.218.22,ping,28095696,240,1790862989


3. Validação

In [18]:
def validar_pares(df_raw, dados_json):
    por_sonda = df_raw.groupby("prb_id").size().sort_values(ascending=False)
    print("Registros por sonda (par):")
    print(por_sonda)

    duplicatas = df_raw.duplicated(subset=["prb_id", "timestamp"]).sum()
    sondas_timeout_total = (df_raw["rcvd"] == 0).sum()
    pacotes_perdidos = sum(1 for r in dados_json for p in r.get("result", []) if "x" in p)

    print("\nPares distintos:", df_raw["prb_id"].nunique())
    print("Duplicatas:", duplicatas)
    print("Registros com timeout total:", sondas_timeout_total)
    print("Pacotes individuais perdidos:", pacotes_perdidos)

    return por_sonda


contagem_por_par = validar_pares(resultado_pares["df_raw"], resultado_pares["dados_json"])

Registros por sonda (par):
prb_id
6330    2520
6332    2520
6251    2519
6313    2519
6323    2519
6329    2519
6333    2518
6331    2518
6324    2518
6342    2518
6317    2517
6316    2517
6326    2517
6327    2517
6328    2517
6320    2516
6325    2515
6336    2510
6335    2487
6334    2450
dtype: int64

Pares distintos: 20
Duplicatas: 0
Registros com timeout total: 44
Pacotes individuais perdidos: 269


4. Transformar para o Contrato de Dados e salvar o CSV final

In [19]:
def transformar_para_contrato(dados_json, destino_fqdn):
    registros = []
    for sonda in dados_json:
        enviados, recebidos = sonda["sent"], sonda["rcvd"]
        perda_pct = ((enviados - recebidos) / enviados) * 100 if enviados > 0 else 100
        rtts = [r["rtt"] for r in sonda.get("result", []) if "rtt" in r]
        if len(rtts) > 1:
            jitter = sum(abs(rtts[i] - rtts[i-1]) for i in range(1, len(rtts))) / (len(rtts) - 1)
        else:
            jitter = 0.0

        registros.append({
            "timestamp": pd.to_datetime(sonda["timestamp"], unit="s", utc=True),
            "par_id": f"{sonda['prb_id']}->{destino_fqdn}",
            "prb_id": sonda["prb_id"],
            "ip": sonda["src_addr"],
            "latencia_ms": round(sonda.get("avg", 0), 2),
            "perda_pacotes_pct": round(perda_pct, 2),
            "jitter_ms": round(jitter, 2),
        })
    return pd.DataFrame(registros)


def classificar_status_relativo(df):
    estatisticas = (
        df[df["perda_pacotes_pct"] == 0]
        .groupby("par_id")["latencia_ms"]
        .agg(latencia_media="mean", latencia_desvio="std")
    )
    df = df.merge(estatisticas, on="par_id", how="left")

    def status(linha):
        if linha["perda_pacotes_pct"] > 0:
            return "FALHA"
        limite_risco = linha["latencia_media"] + linha["latencia_desvio"]
        limite_falha = linha["latencia_media"] + 3 * linha["latencia_desvio"]
        if linha["latencia_ms"] >= limite_falha:
            return "FALHA"
        elif linha["latencia_ms"] >= limite_risco:
            return "RISCO"
        else:
            return "OK"

    df["status_real"] = df.apply(status, axis=1)
    return df.drop(columns=["latencia_media", "latencia_desvio"])


df_contrato = transformar_para_contrato(resultado_pares["dados_json"], anchor_fqdn)
df_contrato = classificar_status_relativo(df_contrato)

caminho_csv_final = PASTA_RAW / "dataset_pares_contrato.csv"
df_contrato.to_csv(caminho_csv_final, index=False, encoding="utf-8")
print("CSV final salvo em:", caminho_csv_final)
df_contrato["status_real"].value_counts()

CSV final salvo em: /content/drive/MyDrive/ripe_atlas/raw/dataset_pares_contrato.csv


,count
status_real,
OK,46955
RISCO,2562
FALHA,734
